# 从零实现 N-BEATS：Backcast、Forecast 与可解释 Basis

N-BEATS 用一串 block 反复解释输入历史：每个 block 输出 backcast，从 residual 中扣除；同时输出 forecast 并累加。本 Notebook 手写 generic/trend/seasonality basis、残差堆叠、严格时间窗口、直接多步预测、评估与发布 wrapper。

合成趋势+双季节序列只验证结构能学习已知模式，不代表真实需求预测。生产系统还要处理缺失、节假日、协变量、层级一致性、概念漂移和概率区间。

In [ ]:
import copy,hashlib,io,json,math,random,warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。
SEED65=6501  # 计算并保存当前步骤的中间状态。
random.seed(SEED65); np.random.seed(SEED65); torch.manual_seed(SEED65); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
def canonical65(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha65(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert torch.get_num_threads()==1  # 用受控断言验证关键不变量。

## 1. 严格时间切分与窗口

序列 $y_t=0.004t+\sin(2\pi t/24)+0.3\sin(2\pi t/7)+\epsilon_t$。前 360 点 train，接着 96 点 validation，最后 96 点 test。normalizer 只用 train。

backcast 长 36、forecast 长 6。一个窗口允许读取 forecast 起点以前的历史，但所有 target 必须完整落在所属 split；禁止随机打散后切分造成未来泄漏。

In [ ]:
def make_series65(length=552,seed=SEED65+1):  # 定义本节可复用的核心函数。
    g=torch.Generator().manual_seed(seed); t=torch.arange(length,dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
    return .004*t+torch.sin(2*math.pi*t/24)+.3*torch.sin(2*math.pi*t/7)+.04*torch.randn(length,generator=g)  # 返回当前分支计算出的结果。
raw65=make_series65(); train_end65=360; val_end65=456; mean65=raw65[:train_end65].mean(); std65=raw65[:train_end65].std(unbiased=False)  # 计算并保存当前步骤的中间状态。
series65=(raw65-mean65)/std65; BACK65=36; HORIZON65=6  # 计算并保存当前步骤的中间状态。
def windows65(start,end):  # 定义本节可复用的核心函数。
    xs=[]; ys=[]; origins=[]  # 计算并保存当前步骤的中间状态。
    for origin in range(max(start,BACK65),end-HORIZON65+1):  # 遍历输入元素以累积或检查结果。
        xs.append(series65[origin-BACK65:origin]); ys.append(series65[origin:origin+HORIZON65]); origins.append(origin)  # 执行当前语句以推进本节示例。
    return torch.stack(xs),torch.stack(ys),torch.tensor(origins)  # 返回当前分支计算出的结果。
train_x65,train_y65,train_o65=windows65(BACK65,train_end65); val_x65,val_y65,val_o65=windows65(train_end65,val_end65); test_x65,test_y65,test_o65=windows65(val_end65,len(raw65))  # 计算并保存当前步骤的中间状态。
assert train_x65.shape[1:]==(BACK65,) and train_y65.shape[1:]==(HORIZON65,)  # 用受控断言验证关键不变量。
assert train_o65.max()+HORIZON65<=train_end65 and val_o65.min()>=train_end65 and test_o65.min()>=val_end65  # 用受控断言验证关键不变量。
assert torch.allclose(raw65,make_series65()) and std65>0  # 用受控断言验证关键不变量。

## 2. Generic、Trend 与 Seasonality basis

block 的 MLP 输出系数 $\theta$，basis 把系数变为 backcast/forecast。Generic 直接把前后系数视为序列；trend 使用归一化时间的多项式；seasonality 使用 Fourier sin/cos。

可解释性来自限制 basis，而不是给任意 MLP 输出贴“趋势”标签。shape 合同分别是 `[B,theta_dim] -> ([B,36],[B,6])`。

In [ ]:
class GenericBasis65(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,backcast,forecast): super().__init__(); self.backcast=backcast; self.forecast=forecast; self.theta_dim=backcast+forecast  # 定义本节可复用的核心函数。
    def forward(self,theta):  # 定义本节可复用的核心函数。
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("generic_theta_contract")  # 按当前条件选择后续控制路径。
        return theta[:,:self.backcast],theta[:,self.backcast:]  # 返回当前分支计算出的结果。
class TrendBasis65(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,backcast,forecast,degree=2):  # 定义本节可复用的核心函数。
        super().__init__(); self.order=degree+1; self.theta_dim=2*self.order  # 计算并保存当前步骤的中间状态。
        tb=torch.linspace(-1,0,backcast); tf=torch.linspace(0,1,forecast)  # 计算并保存当前步骤的中间状态。
        self.register_buffer("back_basis",torch.stack([tb**i for i in range(self.order)])); self.register_buffer("fore_basis",torch.stack([tf**i for i in range(self.order)]))  # 执行当前语句以推进本节示例。
    def forward(self,theta):  # 定义本节可复用的核心函数。
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("trend_theta_contract")  # 按当前条件选择后续控制路径。
        b,f=theta.chunk(2,-1); return b@self.back_basis,f@self.fore_basis  # 计算并保存当前步骤的中间状态。
class SeasonalityBasis65(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,backcast,forecast,harmonics=3,period=24.):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if harmonics<1 or not math.isfinite(period) or period<=0: raise ValueError("seasonality_config_contract")  # 按当前条件选择后续控制路径。
        self.harmonics=harmonics; self.period=float(period); self.theta_dim=4*harmonics  # 计算并保存当前步骤的中间状态。
        def basis(times):  # 定义本节可复用的核心函数。
            freq=torch.arange(1,harmonics+1,dtype=torch.float32)[:,None]  # 计算并保存当前步骤的中间状态。
            phase=2*math.pi*freq*times[None,:]/self.period  # 计算并保存当前步骤的中间状态。
            return torch.cat([torch.cos(phase),torch.sin(phase)],0)  # 返回当前分支计算出的结果。
        self.register_buffer("back_basis",basis(torch.arange(-backcast,0,dtype=torch.float32))); self.register_buffer("fore_basis",basis(torch.arange(forecast,dtype=torch.float32)))  # 计算并保存当前步骤的中间状态。
    def forward(self,theta):  # 定义本节可复用的核心函数。
        if theta.ndim!=2 or theta.shape[1]!=self.theta_dim: raise ValueError("season_theta_contract")  # 按当前条件选择后续控制路径。
        b,f=theta.chunk(2,-1); return b@self.back_basis,f@self.fore_basis  # 计算并保存当前步骤的中间状态。
trend_probe65=TrendBasis65(3,3,1); theta_probe65=torch.tensor([[2.,1.,3.,-1.]])  # 计算并保存当前步骤的中间状态。
back_probe65,fore_probe65=trend_probe65(theta_probe65)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(back_probe65,torch.tensor([[1.,1.5,2.]])) and torch.allclose(fore_probe65,torch.tensor([[3.,2.5,2.]]))  # 用受控断言验证关键不变量。
generic_probe65=GenericBasis65(2,1); assert generic_probe65(torch.tensor([[1.,2.,3.]]))[1].item()==3  # 计算并保存当前步骤的中间状态。
season_probe65=SeasonalityBasis65(3,2,1,period=4.); season_theta65=torch.tensor([[1.,0.,1.,0.]])  # 计算并保存当前步骤的中间状态。
season_back65,season_fore65=season_probe65(season_theta65)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(season_back65,torch.cos(2*math.pi*torch.arange(-3,0)/4.)[None,:],atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(season_fore65,torch.cos(2*math.pi*torch.arange(2)/4.)[None,:],atol=1e-6)  # 用受控断言验证关键不变量。

## 3. Block 与双残差堆叠

每个 block 用四层 MLP 将 residual `[B,36]` 映射到 theta，再由 basis 输出 `(backcast,forecast)`。模型递推：
$$r_{l+1}=r_l-\hat x_l,\qquad \hat y=\sum_l\hat y_l.$$

这称为 doubly residual stacking：输入残差向前传，forecast 残差横向累加。返回各 block component 便于诊断。

In [ ]:
class NBeatsBlock65(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,input_size,basis,hidden=64):  # 定义本节可复用的核心函数。
        super().__init__(); self.input_size=input_size; self.basis=basis  # 计算并保存当前步骤的中间状态。
        self.mlp=nn.Sequential(nn.Linear(input_size,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,basis.theta_dim))  # 计算并保存当前步骤的中间状态。
    def forward(self,x):  # 定义本节可复用的核心函数。
        if x.ndim!=2 or x.shape[1]!=self.input_size or not torch.isfinite(x).all(): raise ValueError("block_input_contract")  # 按当前条件选择后续控制路径。
        return self.basis(self.mlp(x))  # 返回当前分支计算出的结果。
class NBeats65(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,blocks,forecast_size): super().__init__(); self.blocks=nn.ModuleList(blocks); self.forecast_size=forecast_size  # 定义本节可复用的核心函数。
    def forward(self,x,return_components=False):  # 定义本节可复用的核心函数。
        residual=x; forecast=torch.zeros(x.shape[0],self.forecast_size,dtype=x.dtype,device=x.device); components=[]  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            back,fore=block(residual); residual=residual-back; forecast=forecast+fore; components.append(fore)  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(residual).all() or not torch.isfinite(forecast).all(): raise ValueError("nonfinite_nbeats_output")  # 按当前条件选择后续控制路径。
        return (forecast,residual,components) if return_components else forecast  # 返回当前分支计算出的结果。
zero_basis65=GenericBasis65(BACK65,HORIZON65); zero_block65=NBeatsBlock65(BACK65,zero_basis65,8)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for p in zero_block65.parameters(): p.zero_()  # 遍历输入元素以累积或检查结果。
zero_model65=NBeats65([zero_block65],HORIZON65); zf65,zr65,zc65=zero_model65(train_x65[:2],True)  # 计算并保存当前步骤的中间状态。
assert torch.equal(zf65,torch.zeros_like(zf65)) and torch.equal(zr65,train_x65[:2]) and len(zc65)==1  # 用受控断言验证关键不变量。

## 4. 可解释 stack 的受控训练

使用 trend、seasonality、generic 三个 block。训练只最小化 train 窗口 MSE；每 40 步查看 validation，但 test 最后一次报告。基线是把历史最后一个值复制 6 步。

真实 N-BEATS 常用更深 stack、不同 loss 和 ensemble。本例缩小网络以在 CPU 快速验证 residual 与 basis。

In [ ]:
torch.manual_seed(SEED65)  # 执行当前语句以推进本节示例。
model65=NBeats65([NBeatsBlock65(BACK65,TrendBasis65(BACK65,HORIZON65,2)),NBeatsBlock65(BACK65,SeasonalityBasis65(BACK65,HORIZON65,4)),NBeatsBlock65(BACK65,GenericBasis65(BACK65,HORIZON65))],HORIZON65)  # 计算并保存当前步骤的中间状态。
opt65=torch.optim.Adam(model65.parameters(),lr=3e-3); initial65=float(F.mse_loss(model65(val_x65),val_y65)); history65=[]  # 计算并保存当前步骤的中间状态。
for step65 in range(440):  # 遍历输入元素以累积或检查结果。
    pred65=model65(train_x65); loss65=F.mse_loss(pred65,train_y65); opt65.zero_grad(set_to_none=True); loss65.backward(); torch.nn.utils.clip_grad_norm_(model65.parameters(),5.); opt65.step()  # 计算并保存当前步骤的中间状态。
    if step65%40==0: history65.append(float(F.mse_loss(model65(val_x65),val_y65)))  # 按当前条件选择后续控制路径。
with torch.no_grad(): val_pred65=model65(val_x65); test_pred65=model65(test_x65)  # 在受管理的上下文中执行操作。
val_mae65=float((val_pred65-val_y65).abs().mean()); test_mae65=float((test_pred65-test_y65).abs().mean()); naive_mae65=float((test_x65[:,-1,None]-test_y65).abs().mean())  # 计算并保存当前步骤的中间状态。
assert val_mae65<.22 and test_mae65<naive_mae65*.65 and history65[-1]<initial65  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model65.parameters())  # 用受控断言验证关键不变量。
print({"val_mae":round(val_mae65,4),"test_mae":round(test_mae65,4),"naive":round(naive_mae65,4)})  # 执行当前语句以推进本节示例。

## 5. 组件、尺度与未来干预 oracle

component 和应等于总 forecast；改变一个样本不会影响其他 batch 行。发布接口必须接收原始单位 36 点历史，在内部使用 train-only mean/std，再把 6 步输出反标准化。

时间模型的“未来隔离”首先由窗口 builder 保证；网络只看到固定 backcast，不存在把 target 拼入特征的通道。

In [ ]:
with torch.no_grad(): forecast65,residual65,components65=model65(test_x65[:4],True)  # 在受管理的上下文中执行操作。
assert torch.allclose(torch.stack(components65).sum(0),forecast65,atol=1e-6) and residual65.shape==(4,BACK65)  # 用受控断言验证关键不变量。
changed65=test_x65[:4].clone(); changed65[0]+=1.; changed_pred65=model65(changed65)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(changed_pred65[1:],forecast65[1:],atol=1e-6) and not torch.allclose(changed_pred65[0],forecast65[0])  # 用受控断言验证关键不变量。
try: model65(torch.zeros(2,BACK65-1)); raise AssertionError("wrong history accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="block_input_contract"  # 捕获预期异常并验证失败分支。

### 5.1 滚动原点指标、逐 horizon 误差与代数 oracle

多步预测不能只汇报一个平均数：第 1 步很好、第 6 步崩溃时，整体 MAE 会掩盖问题。下面给出逐 horizon MAE、训练集一步朴素尺度上的 MASE，以及中位数 pinball loss。MASE 的分母只能从训练区间计算，否则仍然属于评估泄漏。

同时手工重放每个 block 的 `residual -= backcast` 与 `forecast += component`，验证框架前向确实实现了论文公式；再用 batch 置换证明样本之间没有隐式串扰。

In [ ]:
horizon_mae65=(test_pred65-test_y65).abs().mean(0)  # 计算并保存当前步骤的中间状态。
naive_horizon_mae65=(test_x65[:,-1,None]-test_y65).abs().mean(0)  # 计算并保存当前步骤的中间状态。
train_naive_scale65=(series65[1:train_end65]-series65[:train_end65-1]).abs().mean()  # 计算并保存当前步骤的中间状态。
mase65=(test_pred65-test_y65).abs().mean()/train_naive_scale65  # 计算并保存当前步骤的中间状态。
error65=test_y65-test_pred65; pinball50_65=torch.maximum(.5*error65,-.5*error65).mean()  # 计算并保存当前步骤的中间状态。
assert horizon_mae65.shape==(HORIZON65,) and naive_horizon_mae65.shape==(HORIZON65,)  # 用受控断言验证关键不变量。
assert torch.isfinite(horizon_mae65).all() and bool((horizon_mae65>=0).all())  # 用受控断言验证关键不变量。
assert torch.isclose(horizon_mae65.mean(),torch.tensor(test_mae65),atol=1e-7)  # 用受控断言验证关键不变量。
assert train_naive_scale65>0 and torch.isfinite(mase65) and mase65>0  # 用受控断言验证关键不变量。
assert torch.allclose(pinball50_65,.5*(test_pred65-test_y65).abs().mean(),atol=1e-7)  # 用受控断言验证关键不变量。
manual_residual65=test_x65[:5].clone(); manual_forecast65=torch.zeros(5,HORIZON65)  # 计算并保存当前步骤的中间状态。
for block65 in model65.blocks:  # 遍历输入元素以累积或检查结果。
    manual_back65,manual_fore65=block65(manual_residual65); manual_residual65-=manual_back65; manual_forecast65+=manual_fore65  # 计算并保存当前步骤的中间状态。
api_forecast65,api_residual65,_=model65(test_x65[:5],True)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(manual_forecast65,api_forecast65,atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(manual_residual65,api_residual65,atol=1e-6)  # 用受控断言验证关键不变量。
perm65=torch.tensor([3,0,4,1,2])  # 计算并保存当前步骤的中间状态。
assert torch.allclose(model65(test_x65[:5][perm65]),api_forecast65[perm65],atol=1e-6)  # 用受控断言验证关键不变量。

## 6. Published forecaster

manifest 绑定公式/seed、时间边界、window、normalizer、basis 顺序、训练 recipe 与测试协议。loader 重新生成序列并推导 train mean/std。`PublishedNBeats65.forecast` 只接受原始单位 `[B,36]`，返回原单位 `[B,6]`。

包外 registry 拒绝整体重签；state 摘要包含 key/dtype/shape/bytes。

In [ ]:
def th65(t):  # 定义本节可复用的核心函数。
    v=t.detach().cpu().contiguous(); return sha65(str(v.dtype).encode()+canonical65(list(v.shape)).encode()+v.numpy().tobytes())  # 计算并保存当前步骤的中间状态。
def sd65(state):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for k,v in sorted(state.items()): h.update(k.encode()); h.update(th65(v).encode())  # 遍历输入元素以累积或检查结果。
    return h.hexdigest()  # 返回当前分支计算出的结果。
config65={"input_size":BACK65,"forecast_size":HORIZON65,"blocks":[["trend",2],["seasonality",4],["generic",0]],"seasonality_base_period":24.,"hidden":64}  # 计算并保存当前步骤的中间状态。
manifest65={"artifact_id":"nbeats-synthetic-v1","version":1,"model_config":config65,"data":{"length":552,"seed":SEED65+1,"train_end":train_end65,"val_end":val_end65,"snapshot":th65(raw65)},"preprocess":{"mean":float(mean65),"std":float(std65)},"training":{"seed":SEED65,"optimizer":"Adam","steps":440,"lr":.003,"grad_clip":5.,"validation_interval":40}}  # 计算并保存当前步骤的中间状态。
def package65(model,m):  # 定义本节可复用的核心函数。
    b=io.BytesIO(); torch.save(model.state_dict(),b); raw=b.getvalue(); state=torch.load(io.BytesIO(raw),map_location="cpu",weights_only=True); ms=sha65(canonical65(m).encode()); ss=sd65(state); rs=sha65(raw); bd=sha65(canonical65([ms,ss,rs]).encode()); return {"manifest":copy.deepcopy(m),"manifest_sha":ms,"state_bytes":raw,"state_digest":ss,"state_bytes_sha":rs,"bundle_digest":bd}  # 计算并保存当前步骤的中间状态。
pkg65=package65(model65,manifest65); REG65=MappingProxyType({("nbeats-synthetic-v1",1):pkg65["bundle_digest"]})  # 计算并保存当前步骤的中间状态。
def make_model65(): return NBeats65([NBeatsBlock65(BACK65,TrendBasis65(BACK65,HORIZON65,2)),NBeatsBlock65(BACK65,SeasonalityBasis65(BACK65,HORIZON65,4)),NBeatsBlock65(BACK65,GenericBasis65(BACK65,HORIZON65))],HORIZON65)  # 定义本节可复用的核心函数。
class PublishedNBeats65:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,model,mean,std): self._model=model; self._mean=float(mean); self._std=float(std)  # 定义本节可复用的核心函数。
    @torch.no_grad()  # 为下方定义附加声明式配置。
    def forecast(self,history):  # 定义本节可复用的核心函数。
        x=torch.as_tensor(history,dtype=torch.float32)  # 计算并保存当前步骤的中间状态。
        if x.ndim!=2 or x.shape[0]<1 or x.shape[1]!=BACK65 or not torch.isfinite(x).all(): raise ValueError("published_history_contract")  # 按当前条件选择后续控制路径。
        return self._model((x-self._mean)/self._std)*self._std+self._mean  # 返回当前分支计算出的结果。
def load65(pkg):  # 定义本节可复用的核心函数。
    m=pkg["manifest"]; key=(m.get("artifact_id"),m.get("version"))  # 计算并保存当前步骤的中间状态。
    regen=make_series65(m["data"]["length"],m["data"]["seed"]); dm=regen[:m["data"]["train_end"]].mean(); ds=regen[:m["data"]["train_end"]].std(unbiased=False)  # 计算并保存当前步骤的中间状态。
    if m!=manifest65 or th65(regen)!=m["data"]["snapshot"] or not torch.isclose(dm,torch.tensor(m["preprocess"]["mean"])) or not torch.isclose(ds,torch.tensor(m["preprocess"]["std"])): raise RuntimeError("manifest_data_contract")  # 按当前条件选择后续控制路径。
    current_ms=sha65(canonical65(m).encode()); current_rs=sha65(pkg["state_bytes"])  # 计算并保存当前步骤的中间状态。
    if current_ms!=pkg["manifest_sha"] or current_rs!=pkg["state_bytes_sha"]: raise RuntimeError("state_bytes_contract")  # 按当前条件选择后续控制路径。
    state=torch.load(io.BytesIO(pkg["state_bytes"]),map_location="cpu",weights_only=True)  # 计算并保存当前步骤的中间状态。
    current_sd=sd65(state)  # 计算并保存当前步骤的中间状态。
    if current_sd!=pkg["state_digest"]: raise RuntimeError("state_digest_contract")  # 按当前条件选择后续控制路径。
    current_bundle=sha65(canonical65([current_ms,current_sd,current_rs]).encode())  # 计算并保存当前步骤的中间状态。
    if pkg.get("bundle_digest")!=current_bundle: raise RuntimeError("bundle_contract")  # 按当前条件选择后续控制路径。
    if REG65.get(key)!=current_bundle: raise RuntimeError("publisher_registry_rejected")  # 按当前条件选择后续控制路径。
    model=make_model65(); model.load_state_dict(state); model.eval(); return PublishedNBeats65(model,dm,ds)  # 计算并保存当前步骤的中间状态。
pub65=load65(pkg65); raw_history65=raw65[test_o65[:3,None]-BACK65+torch.arange(BACK65)]  # 计算并保存当前步骤的中间状态。
served65=pub65.forecast(raw_history65); direct65=model65(test_x65[:3])*std65+mean65  # 计算并保存当前步骤的中间状态。
assert torch.allclose(served65,direct65,atol=1e-6)  # 用受控断言验证关键不变量。
forged65=package65(make_model65(),manifest65)  # 计算并保存当前步骤的中间状态。
try: load65(forged65); raise AssertionError("re-signed forecaster accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="publisher_registry_rejected"  # 捕获预期异常并验证失败分支。
forged_old_bundle65=copy.deepcopy(forged65); forged_old_bundle65["bundle_digest"]=pkg65["bundle_digest"]  # 计算并保存当前步骤的中间状态。
try: load65(forged_old_bundle65); raise AssertionError("forged forecaster with old bundle accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="bundle_contract"  # 捕获预期异常并验证失败分支。

### 6.1 服务负例与篡改测试

一个可发布预测器必须把失败行为也定义清楚：空历史、错长度和 NaN 要拒绝；同一输入要得到逐位一致的结果；manifest 或权重字节被修改时必须 fail closed。这里只信任代码外的只读 registry，因此攻击者即使替换权重并重新计算包内摘要，也不能把它伪装成已批准版本。

生产系统还应把告警阈值按 horizon 拆开，并记录输入缺失率、漂移、预测延迟和回填后的真实误差，避免只监控平均 loss。

In [ ]:
assert served65.shape==(3,HORIZON65) and served65.dtype==torch.float32 and torch.isfinite(served65).all()  # 用受控断言验证关键不变量。
assert torch.equal(served65,pub65.forecast(raw_history65))  # 用受控断言验证关键不变量。
assert isinstance(REG65,MappingProxyType) and len(REG65)==1  # 用受控断言验证关键不变量。
try: pub65.forecast(torch.zeros(2,BACK65-1)); raise AssertionError("wrong published window accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="published_history_contract"  # 捕获预期异常并验证失败分支。
try: pub65.forecast(torch.full((1,BACK65),float("nan"))); raise AssertionError("NaN history accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="published_history_contract"  # 捕获预期异常并验证失败分支。
try: pub65.forecast(torch.empty(0,BACK65)); raise AssertionError("empty history batch accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="published_history_contract"  # 捕获预期异常并验证失败分支。
tampered_manifest65=copy.deepcopy(pkg65); tampered_manifest65["manifest"]["training"]["steps"]+=1  # 计算并保存当前步骤的中间状态。
try: load65(tampered_manifest65); raise AssertionError("tampered manifest accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="manifest_data_contract"  # 捕获预期异常并验证失败分支。
damaged_bytes65=copy.deepcopy(pkg65); damaged_bytes65["state_bytes"]=damaged_bytes65["state_bytes"]+b"x"  # 计算并保存当前步骤的中间状态。
try: load65(damaged_bytes65); raise AssertionError("damaged bytes accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="state_bytes_contract"  # 捕获预期异常并验证失败分支。
state_copy65={k:v.clone() for k,v in model65.state_dict().items()}; first_key65=next(iter(state_copy65)); state_copy65[first_key65].view(-1)[0]+=1  # 计算并保存当前步骤的中间状态。
assert sd65(state_copy65)!=sd65(model65.state_dict())  # 用受控断言验证关键不变量。

## 7. 失败模式、复杂度与来源

常见错误：随机切分窗口；全数据标准化；forecast target 越过 split；把 component 当因果分解；漏减 backcast；递归多步却按直接多步评估；只和零基线比较。MLP 计算约随 block 数与 hidden 平方增长，窗口数据复制也可能成为内存瓶颈。

- Oreshkin et al., [N-BEATS](https://arxiv.org/abs/1905.10437), ICLR 2020。
- Olivares et al., [NeuralForecast](https://arxiv.org/abs/2202.12852)，工程化复现背景。
- Hyndman & Athanasopoulos, [Forecasting: Principles and Practice](https://otexts.com/fpp3/)，时间评估背景。